<pre>
#+TITLE: Project 2: Analyzing the 2008 Financial Crisis
#+AUTHOR: Edgar Huamantla
#+DATE: 2026-03-22
</pre>

In [ ]:
# Because jupyter notebook has a global name space; use this area for imports
import pandas as pd
import numpy as np
import plotly.express as px
import math
from typing import Union

## SQLAlchemy imports
from sqlalchemy import create_engine
from sqlalchemy.types import Date, Float, String, Integer, Boolean

# Obtain resource files

In [ ]:
import pathlib
import importlib.util

#check if gdown is installed in current python environment; alert the user to activate their virtual environment to avoid installing to global
if importlib.util.find_spec("gdown") is None:
    print("gdown is not installed. Please activate your virtual environment and install gdown to proceed.")

# dictionary: filename and gdown id
file_names = {'USE3L0712.RSK': '1Y0WE4cqbZfdNEvFuWIDR_d6oDbHHNkly',
              'USE3L0812.RSK': '1Z3vdd5m8uoB4whomq92DQbZHJiKYJc0v'}

curr_working_dir = pathlib.Path.cwd()
print(f"Current working directory: {curr_working_dir}")

for file_name, gdown_id in file_names.items():
    file_path = curr_working_dir / file_name
    if not file_path.exists():
        print(f"File {file_path} does not exist.")
        !gdown {gdown_id} -O {file_path}
    else:
        print(f"File {file_path} already exists. No need to download.")




In [ ]:
# check file integrity of downloaded files
for file_count, file_name in enumerate(file_names.keys(), start=1):
    print(f"\nChecking integrity of file {file_count}: {file_name}...")
    ! head -n 5 {file_name}

# Extract 
 - parse text file into panadas file
 - ignore first row with date ; seen in the head output
    - set second row to header

In [ ]:
barra_df_07 = pd.read_csv('USE3L0712.RSK', skiprows=1, header=0)
print(f"\nDataframe shape: {barra_df_07.shape}")
print("head")
display(barra_df_07.head(3))
print("tail")
display(barra_df_07.tail(3))

In [ ]:
# print DataFrame information showing index dtype and columns, non-NA values and memory usage.
barra_df_07.info()

In [ ]:
# show column names
barra_df_07.columns

In [ ]:
# clean column names; first remove leading and trailing white spaces
barra_df_07.columns = barra_df_07.columns.str.strip()

# check if the column name has invalid characters
invalid_characters = {"%": "_pct"}

# check if column begins with a number; if so, prepend with "col_"
def check_name_leads_with_number(column_name):
    if column_name[0].isdigit():
        print(f"Column name '{column_name}' begins with a number. Prepending with 'col_'.")
        return "col_" + column_name
    else:
        return column_name

# clean column names; replace invalid characters and check if column name begins with a number
for invalid_char, replacement in invalid_characters.items():
    barra_df_07.columns = barra_df_07.columns.str.replace(invalid_char, replacement, regex=False)
barra_df_07.columns = barra_df_07.columns.map(check_name_leads_with_number)

# make lowercase
barra_df_07.columns = barra_df_07.columns.str.lower()


In [ ]:
# new column names
barra_df_07.columns

In [ ]:
# Descriptive statistics include those that summarize the central tendency, dispersion and shape of a dataset’s distribution, excluding NaN values.
barra_df_07.describe()

In [ ]:
# box plot numeric columns; find them first

def find_numeric_columns(df):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"found {len(numeric_cols)} numeric columns in the DataFrame.")
    print(f"Numeric columns: {numeric_cols}")
    
    return numeric_cols

numeric_columns = find_numeric_columns(barra_df_07)

# to fix the the stack of boxplots we define rows
num_rows = math.ceil(len(numeric_columns) / 4) # 4 columns per row


# melt the df; so that it is not on the same axis
df_melted = barra_df_07.melt(
                            value_vars=numeric_columns,
                            var_name="factor_type", 
                            value_name="factor_value"
                            )


fig = px.box(
    df_melted,
    y="factor_value", # numeric column to plot
    facet_col="factor_type", # categorical column to facet by
    facet_col_wrap=4, # like a line break 
    height = 300 * num_rows, # adjust height based on number of rows needed
    template='plotly_dark',
    points='outliers', # show outliers as points
    facet_col_spacing=0.02, # reduce spacing between facets
    facet_row_spacing=0.02, # reduce spacing between rows of facets
)

fig.update_yaxes(matches=None, showticklabels=True) # allow y axes to have different scales



#clean title
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))


#fig.show()

# Transform (Data Cleaning)
 - column names - trim leading and trailing white spaces -- done earlier
    - (optional): set make column names lowercase
 - use a dictionary to replace invalid characters and the value with its replacement (i.e. {% : "_pct"})
 - identify how NaN values are encoded (from box plot hbta has -999)
 - set index

In [ ]:
barra_df_07.head(3)

In [ ]:
# display current indices
barra_df_07.index

In [ ]:
barra_df_07.set_index('ticker').index # by itself does not persist, need to assign back to df or use inplace=True
barra_df_07 = barra_df_07.set_index('ticker') # assign back to df to persist the change

In [ ]:
# find non numbers by column
# numeric_columns already holds numeric columns
non_numeric_columns = [col for col in barra_df_07.columns if col not in numeric_columns]
non_numeric_columns

#optionally: barra_df_07.select_dtypes('object')

In [ ]:
# barra_df_07.loc['AAPL'] error
# barra_df_07[non_numeric_columns].str.strip() # does not work because str returns a series. need to apply to columns separately

def strip_value(value: pd.Series) -> str:
    return value.str.strip()

barra_df_07[non_numeric_columns] = barra_df_07[non_numeric_columns].apply(strip_value)

# lambda way
# barra_df_07[non_numeric_columns] = barra_df_07[non_numeric_columns].apply(lambda x: x.str.strip())

In [ ]:
barra_df_07.head(3)

In [ ]:
print(barra_df_07.index[:5])
print(barra_df_07.index.name)

In [ ]:
# apply strip function to index
barra_df_07.index = barra_df_07.index.str.strip()
barra_df_07.index[:5]

### Troubleshooting key error
```python
# look at first/last labels of the index
print(df.index[:5])
# check if ticket is the index
print(df.index.name)
```

```text
Index(['IX    ', 'SAOL  ', 'IXYS  ', 'CDGT  ', 'CVRG  '], dtype='object', name='ticker')
ticker
```

white spaces in ticker

confirm with substring search
```python
aapl_filter = df[df.index.str.contains('AAPL', case=False, na=False)]
aapl_filter.index[:1]
```

```text
#STDOUT
Index(['AAPL  '], dtype='object', name='ticker')
```

In [ ]:
aapl_filter = barra_df_07[barra_df_07.index.str.contains('AAPL', case=False, na=False)]
aapl_filter.index[:1]

In [ ]:
barra_df_07.loc['AAPL']

## look for null values
- -999 is being used to encode null values
    - search in the numeric columns, return the column

In [ ]:
# intially was going to search all columns, but -999 is numeric so we can just search numeric columns for -999
cols_with_nulls = barra_df_07.columns[(barra_df_07 == -999).any()]
cols_with_nulls

In [ ]:
# null counts
null_counts = (barra_df_07['hbta'] == -999).sum()
null_counts

In [ ]:
# replace the -999 with np.nan
barra_df_07['hbta'] = barra_df_07['hbta'].replace(-999, np.nan)

In [ ]:
# earlier we saw the hbta box plot with the -999 values
fig = px.box(
    barra_df_07['hbta'],
    y='hbta',
    points='outliers',
    notched=True,
    template='plotly_dark',
    title='Box plot of hbta after replacing -999 with NaN'
)
fig.show()

### SQL Books/References 
- SQL Queries for Mere Mortals (John Viescas) 
- Using SQLite (Jay Kreibich / O'Reilly) 
- SQLAlchemy 2 in practice (Miguel Grinberg)

# LOAD
- load into SQL

## Understanding SQL
A database exists to collect and store data in some organized manner for a specific purpose.
Generally, there are two types of databases:
- operational : used to collect, modify and maintain data on a day to day basis.
- analytical : stores and tracks historical and time dependent data. 
    - Data tends to be static -- meaning it is never/rarely modified. New data is often added.

The Father of the relational model, is Dr. Edgar F. Codd. A relation is a table in set theory and the Relational model is based on 
two branches of mathematics:
- 1. Set theory
- 2. First-order predicate logic -- Filters

The father of data warehousing was William (Bill) H. Immon who developed the idea that organizations would access data stored in any number of non-relational databases.

## Anatomy of a Relational Database
Data in a relational database is stored in, you guesssed it, relations. Relations are tables composed of tuples and attributes. 
- Tuples are another way of saying records or rows. 
- Attributes are the fields and columns in a table.
    - Columns is a structure that stores data. It represents a characteristic of the subject of the table.

A table should always represent a single specific subject.
Tables can represent:
- an object: tangible; (think noun) -- person, place or thing.
- event: occurs at a given point in time aand has characteristics you wish to record.

We retrieve data from columns, and if they are properly designed, it only contains only one value.
Rows represent a unique instance of the subject of a table.
- a row is composed of the entire set of columns in a table.

Keys: one or more columns that uniquely identify each row within a table.
- A primary key is important for two reasons, in addition to enforcing table-level integrity and helps establish relationships with
other data.
    - 1. its value represents a specific row throughout the entire database
    - 2. it's column identifies a given table throughout the entire database.

Foreign Keys: take a copy of the primary key form the first table and insert into a second table -- becoming a foreign key. They help to ensure relationship level integrity. Rows in both tables will always be properly related because the value of a foreign key must be drawn from the values of the primary key to which it refers to.
- the second table already has a primary key of its own, and the primary key you are introducing from the first table is foreign to the second table.

Views: are virtual tables composed of columns from one or more tables in the databse. 
- views can be thought of a saved query because only the structure is stored.

### Relationships
- one-to-one: a table has been split into two parts.
- one-to-many: the 'one' has a relationship with many rows in the "many" side.
- many-to-many: these require a 'linking table'. Linking tables are a way of associating rows from one table with those of th oether. The advantage is that it allows you to associate any number of rows from both tables in the relationship.

There is a difference between:
- Database theory: rules and theory that form the basis of the relational model
- Database design: structured, organized set of processes used to design a relational database. It works to ensure integrity, consistency, and accuracy of the data.

# SQLALCHEMY

```python
pip install sqlalchemy
```
SQLAlchemy library is divided into two modules: CORE and ORM (Object Relational Mapping).

Core: contains the database integration logic for supported database dialects, collection of classes to describe database tables, and a system for generating SQL statements using Python language constructs.

ORM: an abstraction layer between Python and the database that allows database operations to be derived from actions performed on Python objects.

## The Database Engine:
SQLAlchemy uses 'engine' objects to manage connections to a database. The create_engine() function constructs an engine given a database URL.

## SQLITE
SQLite is a C-language library that implements a self-contained relational database.

In [ ]:
def get_dtype_name(object: pd.DataFrame) -> dict:
    """
    returns name of the dtype given an object
    """
    return object.name
    
type_dict = barra_df_07.dtypes.apply(get_dtype_name).to_dict()
type_dict